# Task 3: Store Cleaned Data in PostgreSQL

## Objective
Design and implement a relational database schema in PostgreSQL to store cleaned and processed bank review data, simulating a real-world data engineering workflow.

## Database Connection


In [2]:
import pandas as pd
import psycopg2
from sqlalchemy import create_engine
DB_USER = "postgres"
DB_PASSWORD = "2018"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "bank_reviews"

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

## Load Cleaned Dataset

In [3]:
df = pd.read_csv("../data/raw/cleaned_reviews.csv")
df.head()

,review,rating,date,bank,source
0,Good,5,2026-05-16,Commercial Bank of Ethiopia,Google Play
1,🤙🏼🤙🏼,5,2026-05-16,Commercial Bank of Ethiopia,Google Play
2,worst,1,2026-05-16,Commercial Bank of Ethiopia,Google Play
3,this app very full,5,2026-05-16,Commercial Bank of Ethiopia,Google Play
4,good apps,4,2026-05-16,Commercial Bank of Ethiopia,Google Play


## Insert Banks Table

In [5]:
banks_df = df[["bank"]].drop_duplicates()

banks_df = banks_df.rename(columns={
    "bank": "bank_name"
})

banks_df["app_name"] = banks_df["bank_name"] + " Mobile App"

banks_df.to_sql(
    "banks",
    engine,
    if_exists="append",
    index=False
)

2

## Retrieve bank_id Mapping

In [6]:
query = "SELECT * FROM banks"
banks = pd.read_sql(query, engine)

bank_map = dict(zip(banks["bank_name"], banks["bank_id"]))

df["bank_id"] = df["bank"].map(bank_map)

## Prepare Reviews Table

In [16]:
df = df.rename(columns={
    "review": "review_text",
    "date": "review_date",
    "bank": "bank_name"
})

df["sentiment_label"] = "positive"  
df["sentiment_score"] = 0.90
df["identified_theme"] = "General"

reviews_df = df[[
    "bank_id",
    "review_text",
    "rating",
    "review_date",
    "sentiment_label",
    "sentiment_score",
    "identified_theme",
    "source"
]].copy()
reviews_df = reviews_df.rename(columns={
    "date": "review_date"
})

## Insert Reviews into PostgreSQL

In [17]:
reviews_df.to_sql(
    "reviews",
    engine,
    if_exists="append",
    index=False
)

print("Inserted reviews successfully.")

Inserted reviews successfully.


## Verification Queries

In [18]:
query = """
SELECT b.bank_name, COUNT(*) AS total_reviews
FROM reviews r
JOIN banks b
ON r.bank_id = b.bank_id
GROUP BY b.bank_name
"""

pd.read_sql(query, engine)

,bank_name,total_reviews
0,Bank of Abyssinia,498
1,Commercial Bank of Ethiopia,482


## Average rating per bank

In [19]:
query = """
SELECT b.bank_name, AVG(r.rating) AS avg_rating
FROM reviews r
JOIN banks b
ON r.bank_id = b.bank_id
GROUP BY b.bank_name
"""

pd.read_sql(query, engine)

,bank_name,avg_rating
0,Bank of Abyssinia,3.562249
1,Commercial Bank of Ethiopia,4.089212


## Null checks

In [20]:
query = """
SELECT
    COUNT(*) FILTER (WHERE review_text IS NULL) AS null_reviews,
    COUNT(*) FILTER (WHERE sentiment_label IS NULL) AS null_sentiment,
    COUNT(*) FILTER (WHERE identified_theme IS NULL) AS null_themes
FROM reviews
"""

pd.read_sql(query, engine)

,null_reviews,null_sentiment,null_themes
0,0,0,0


## Database Design

The PostgreSQL schema consists of two normalized tables:

### banks
Stores metadata about banking applications.

### reviews
Stores processed customer reviews including:
- sentiment results
- identified themes
- ratings
- review source

The tables are connected through a foreign key (`bank_id`) to ensure referential integrity.